In [ ]:
!pip -q install \
    langchain \
    langchain-community \
    langchain-text-splitters \
    langchain-huggingface \
    pypdf \
    sentence-transformers \
    faiss-cpu \
    transformers \
    accelerate

                         ABC Documents HR_Policy_2026.pdf
                              |
                              v
User Question ---> Retriever ---> Relevant Documents
      |                            |
      |                            v
      +----------------------> Prompt
                                   |
                                   v
                                  LLM
                                   |
                                   v
                                Answer

Employees with 0-2 years of service:
15 annual leave days.

Employees with 3-5 years of service:
20 annual leave days.

Employees with more than 5 years:
25 annual leave days.

Use the following company information to answer the question.

Context:
Employees with 3-5 years of service receive
20 annual leave days.

Question:
How many annual leave days do employees get after 3 years?

Answer using only the provided context.

Document chunking
Chunk 1:
Company introduction...

Chunk 2:
Working hours...

Chunk 3:
Annual leave policy...

Chunk 4:
Sick leave policy...

Chunk 5:
Maternity leave policy...

Chunk 6:
Travel reimbursement...

Embedding Generation
"Employees receive 20 days annual leave"
                 |
                 v
          Embedding Model
                 |
                 v
[0.21, -0.53, 0.87, 0.12, ...]

"How many vacation days do employees get?"

"Employees receive 20 days of annual leave."
vacation
annual leave

Vector Database
Vector DB

ID       Vector                    Text
------------------------------------------------
101      [0.2, 0.8, ...]          Annual leave...
102      [0.7, 0.1, ...]          Sick leave...
103      [0.3, 0.9, ...]          Maternity leave...
104      [0.9, 0.2, ...]          VPN configuration...

PDF
 |
 v
Extract Text
 |
 v
Chunk Text
 |
 v
Embedding Model
 |
 v
Vectors
 |
 v
Vector Database

User Question
      |
      v
Embedding Model
      |
      v
Question Vector
[0.23, -0.48, 0.81, ...]

                 Query embedding
             [0.23, 0.50, 0.80, ...]
                        |
                        v
               Similarity Search
                /       |       \
               /        |        \
              v         v         v
          Vector A   Vector B   Vector C
           0.94       0.32       0.89
             |
             v
         Top-K results

"How many annual leave days do I get?"
                  ↓
       text-embedding-3-small
                  ↓
 [0.023, -0.118, 0.847, ...]
                  ↓
             Vector DB

## Upload the practice PDF

Upload `RAG_Practice_Knowledge_Base.pdf` (or another text-based PDF).


In [ ]:
from google.colab import files

uploaded = files.upload()

pdf_path = list(uploaded.keys())[0]

print("Uploaded:", pdf_path)

Saving company_it_policy_rag_15_chunks.pdf to company_it_policy_rag_15_chunks (2).pdf
Uploaded: company_it_policy_rag_15_chunks (2).pdf


In [ ]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader(pdf_path)

documents = loader.load()

print("Pages loaded:", len(documents))

Pages loaded: 4


In [ ]:
print(documents[0].page_content[:1000])

Company IT Support and Security Handbook
1. IT Service Desk
The IT Service Desk is the primary contact for employee technology support. Standard service desk hours are Monday
through Friday from 8:00 AM to 6:00 PM local business time. The desk is closed on weekends and company holidays
except for approved on-call coverage. Employees can contact the desk for account lockouts, password resets, VPN
problems, email issues, software requests, printer failures, Wi-Fi problems, and company laptop troubleshooting.
Critical production or security incidents should be reported through the emergency support channel immediately rather
than waiting for normal service hours.
2. Opening a Support Ticket
When opening an IT ticket, provide enough detail for the technician to start troubleshooting without a long
back-and-forth. Include your name, department, device name or asset tag, office or remote location, a short description
of the problem, the exact error message if one appears, and the approximate

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=700,
    chunk_overlap=100
)

chunks = text_splitter.split_documents(documents)

print("Total chunks:", len(chunks))
print("chunks exists:", "chunks" in globals())

Total chunks: 18
chunks exists: True


In [ ]:
print(chunks[0].page_content)

Company IT Support and Security Handbook
1. IT Service Desk
The IT Service Desk is the primary contact for employee technology support. Standard service desk hours are Monday
through Friday from 8:00 AM to 6:00 PM local business time. The desk is closed on weekends and company holidays
except for approved on-call coverage. Employees can contact the desk for account lockouts, password resets, VPN
problems, email issues, software requests, printer failures, Wi-Fi problems, and company laptop troubleshooting.
Critical production or security incidents should be reported through the emergency support channel immediately rather
than waiting for normal service hours.
2. Opening a Support Ticket


## Embeddings and vector store

The embedding model turns each chunk into a numeric vector. The vector store keeps those vectors and lets us search for chunks that are semantically close to a question.


In [ ]:
!pip install -q "transformers<5" sentencepiece


In [ ]:
!pip uninstall -y transformers sentence-transformers huggingface-hub

!pip install -q \
    "transformers==4.57.6" \
    "sentence-transformers==5.1.0" \
    "huggingface-hub==0.36.0" \
    sentencepiece

Found existing installation: transformers 4.57.6
Uninstalling transformers-4.57.6:
  Successfully uninstalled transformers-4.57.6
Found existing installation: sentence-transformers 5.1.0
Uninstalling sentence-transformers-5.1.0:
  Successfully uninstalled sentence-transformers-5.1.0
Found existing installation: huggingface-hub 0.36.0
Uninstalling huggingface-hub-0.36.0:
  Successfully uninstalled huggingface-hub-0.36.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.0 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.0 which is incompatible.


In [ ]:
import transformers
import sentence_transformers
import huggingface_hub

print("transformers:", transformers.__version__)
print("sentence-transformers:", sentence_transformers.__version__)
print("huggingface-hub:", huggingface_hub.__version__)

transformers: 4.57.6
sentence-transformers: 5.1.0
huggingface-hub: 0.36.0


In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embeddings created!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embeddings created!


In [ ]:
sample_vector = embeddings.embed_query(
    "How do I connect to the company VPN?"
)

print("Vector dimensions:", len(sample_vector))
print(sample_vector[:10])

Vector dimensions: 384
[-0.06029343977570534, -0.011807037517428398, -0.013232690282166004, -0.017646590247750282, -0.044578053057193756, 0.06925076246261597, 0.02983466535806656, -0.004830152727663517, 0.043525345623493195, -0.060746047645807266]


In [ ]:
from langchain_community.vectorstores import FAISS

vector_store = FAISS.from_documents(
    chunks,
    embeddings
)

print("Vector store created successfully")

Vector store created successfully


In [ ]:
question = "What are the IT service desk hours?"

results = vector_store.similarity_search(
    question,
    k=3
)

for i, doc in enumerate(results, 1):
    print(f"\n--- Result {i} ---")
    print(doc.page_content)


--- Result 1 ---
the normal IT request channel. Standard IT Service Desk hours are Monday through Friday from 8:00 AM to 6:00 PM
local business time, with emergency channels reserved for critical operational or security incidents.

--- Result 2 ---
Company IT Support and Security Handbook
1. IT Service Desk
The IT Service Desk is the primary contact for employee technology support. Standard service desk hours are Monday
through Friday from 8:00 AM to 6:00 PM local business time. The desk is closed on weekends and company holidays
except for approved on-call coverage. Employees can contact the desk for account lockouts, password resets, VPN
problems, email issues, software requests, printer failures, Wi-Fi problems, and company laptop troubleshooting.
Critical production or security incidents should be reported through the emergency support channel immediately rather
than waiting for normal service hours.
2. Opening a Support Ticket

--- Result 3 ---
than waiting for normal service hou

In [ ]:
print("vector_store exists:", "vector_store" in globals())

vector_store exists: True


In [ ]:
docs = retriever.invoke(
    "What should an employee do if a laptop is lost?"
)

for doc in docs:
    print(doc.page_content)
    print("-" * 50)

Device Protection and Incident Response
9. Lost or Stolen Laptop
A lost or stolen company laptop must be reported immediately to the IT Service Desk or Security team. Do not wait until
the next business day if the device may contain company information. Provide the asset tag or device name if known,
the approximate time and location where it was last seen, and whether the laptop was powered on or unlocked. IT may
revoke active sessions, disable authentication tokens, block the device, or remotely wipe company data where
supported. If the laptop was stolen, follow company guidance regarding a police report. Employees should never
--------------------------------------------------
company policy. Before returning or replacing a laptop, employees should synchronize required business files to the
approved company storage location and follow IT instructions rather than manually wiping or reinstalling the operating
system.
12. Physical Security
Employees are responsible for reasonable physic

In [ ]:
!pip install -q -U \
    "huggingface-hub>=1.2,<2" \
    "transformers>=5.0,<6" \
    "sentence-transformers>=5.2,<6" \
    sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 611.3/611.3 kB 5.6 MB/s eta 0:00:00


In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/flan-t5-small"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

print("Model loaded successfully!")

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Model loaded successfully!


In [ ]:
question = "What is artificial intelligence?"

inputs = tokenizer(question, return_tensors="pt")

outputs = model.generate(
    **inputs,
    max_new_tokens=100
)

answer = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print(answer)

artificial intelligence


In [ ]:
Model output:
[71, 42, 986, 12, ...]

        ↓ decode

"Artificial intelligence is..."

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/flan-t5-small"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

print("Model loaded!")

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Model loaded!


In [ ]:
retriever = vector_store.as_retriever(
    search_kwargs={"k": 3}
)

print("Retriever created!")

Retriever created!


In [ ]:
def ask_pdf(question):

    # 1. Retrieve relevant chunks from FAISS
    docs = retriever.invoke(question)

    # 2. Combine retrieved chunks
    context = "\n\n".join(
        doc.page_content for doc in docs
    )

    # 3. Build the prompt
    prompt = f"""
Answer the question using only the context below.

If the answer is not present in the context, say:
I cannot find that information in the document.

Context:
{context}

Question:
{question}

Answer:
"""

    # 4. Tokenize the prompt
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    # 5. Generate with FLAN-T5 directly
    outputs = model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=False
    )

    # 6. Convert tokens back to text
    answer = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return answer

In [ ]:
answer = ask_pdf(
    "What are the IT service desk hours?"
)

print(answer)

Monday through Friday from 8:00 AM to 6:00 PM local business time
